<a href="https://colab.research.google.com/github/suriyaksd-dotcom/FAKE-NEWS-DETECTION/blob/main/FAKE_NEWS_DETECTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# 1. Import Libraries

import numpy as np
import pandas as pd
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
# 2. Load Dataset


fake = pd.read_csv("/content/drive/MyDrive/fake news/Fake.csv")
true = pd.read_csv("/content/drive/MyDrive/fake news/real news/True.csv")

In [4]:
fake["label"] = 0
true["label"] = 1

data = pd.concat([fake, true], axis=0)
data = data.sample(frac=1).reset_index(drop=True)

print("Dataset loaded successfully!")
print(data.head())

Dataset loaded successfully!
                                               title  \
0   Angry Cruz Kicks Young Boy From Rally For Bei...   
1  Trump will not be charged with 'inciting riot'...   
2   Doctors Can Lie To Pregnant Women If This Rep...   
3  After ditching Taiwan, China says Panama will ...   
4  White House says tax bill will not hurt Puerto...   

                                                text       subject  \
0  Ted Cruz is just one night away from losing th...          News   
1  WINSTON-SALEM, N.C. (Reuters) - A North Caroli...  politicsNews   
2  Republicans nationwide seem to want to get rid...          News   
3  BEIJING (Reuters) - China will provide Panama ...     worldnews   
4  WASHINGTON (Reuters) - Sweeping tax code chang...  politicsNews   

                 date  label  
0         May 1, 2016      0  
1     March 14, 2016       1  
2       March 9, 2017      0  
3  November 17, 2017       1  
4  December 20, 2017       1  


In [5]:
# -------------------- 3. CLEAN TEXT --------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'\n', '', text)
    return text

print("Cleaning text...")
data["text"] = data["text"].apply(clean_text)

Cleaning text...


In [6]:
# -------------------- 4. TRAIN TEST SPLIT --------------------
X = data["text"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
# -------------------- 5. TOKENIZATION & PADDING --------------------
vocab_size = 20000
max_length = 500

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length)